In [ ]:
from src.slm_initialise import slm_initialise
from src.utils import *
from src.plots import plot_fit,bb,g
from slmsuite.holography.algorithms import FeedbackHologram
from slmsuite.holography.toolbox import convert_vector
from slmsuite.holography.toolbox.phase import zernike_sum,zernike_get_string,ZERNIKE_NAMES 
from src.algos import *
from src.zernike import get_zernike_calibrations,get_cholo_phase,get_cholo_bb_g_phase,fit_metric,fit_metric_udf,plot_parabola,get_zernike_dict_from_metrics_df,get_zernike_metrics_df,zernike_names
from src import global_store
from src.config import base_path
%load_ext autoreload
%autoreload 2

In [ ]:
fs = slm_initialise(force=True,verbose=True)
calibrate(fs,vendor = False,refresh = False)
sp_calibrate(fs,refresh=False)
plot_cam(fs,phase="zero",centre=True, save=True)

In [ ]:
def bb(x, y, A, w, xc, yc, B):
    r = np.sqrt((x - xc)**2 + (y - yc)**2)
    return B + A * (r**4) * np.exp(-(r**2)/(w**2))

def g(x, y, A, w, xc, yc, B):
    r = np.sqrt((x - xc)**2 + (y - yc)**2)
    return B + A * np.exp(-(r**2)/(w**2))

In [ ]:
target_shape = fs.cam.shape
j,i = np.meshgrid(np.arange(target_shape[0]), np.arange(target_shape[1]), indexing='ij')

centrex = 1180
centrey = 230
n_spots = 5
r = 40
n = 4
rad = 100
bb_locations = np.array([[centrex+(i-4)*rad +50 ,centrey + (j-1)*rad] for i in range(6) for j in range(4)]).T
# spot_locations = np.array([[centrex+(i-2)*rad + 50,centrey + (j-1)*rad + 50] for i in range(n) for j in range(n)]).T
spot_locations_list = []
for k in range(bb_locations.T.shape[0]):
    if (k -(k//4))%2 == 1:
        for theta in np.arange(0,2*np.pi,2*np.pi/5):
            spot_locations_list.append([int(bb_locations.T[k][0]+r*np.sin(theta)) , int(bb_locations.T[k][1]-+r*np.cos(theta))])
spot_locations = np.array(spot_locations_list).T
target = np.zeros(target_shape)
for spot_location in spot_locations.T:
    xc, yc = spot_location
    target = target + g(i,j,0.012757392269578168e3,8.997853311827269,xc, yc,0)
for bb_location in bb_locations.T:
    xc, yc = bb_location
    # target = target + bb(i,j,1.30590871e+04,1.30660485e+04,8.56061191e+00,8.54272142e+00,xc,yc)
    target = target + bb(i, j,0.012757392269578168,8.997853311827269, xc, yc, 0)
plt.imshow(target)

In [ ]:
spot_vectors  = bb_locations.T
spot_weights = np.zeros((spot_vectors.shape[0],5))
spot_weights[:,0] = 1
phis = []
phi_nf_gs,phi_ff_gs = gs_spots(spot_vectors,fs,n_iters = 150)

In [ ]:
# phis = ultraprecise_spot_correction(bb_locations.T,fs,phi_ff_gs,n_iters = 60)
# bb_spot_phase = phis[59].numpy().copy()
bb_spot_phase = phi_nf_gs.numpy()
bbs_phase = bb_spot_phase + bottle_beam_phase(fs)
plt.imshow(bbs_phase)

In [ ]:
spot_vectors  = spot_locations.T
spot_weights = np.zeros((spot_vectors.shape[0],5))
spot_weights[:,0] = 1
phis = []
phi_nf_gs,phi_ff_gs = gs(spot_vectors,fs,n_iters = 100)
tweezers_phase = phi_nf_gs.numpy()

In [ ]:
surface_code_phase = random_masking(bbs_phase,tweezers_phase)
plot_cam(fs,phase=surface_code_phase)

# Work in progress

### Make a function get_cholo_bb_g_phase to directly create the bb and tweezer mix by taking as input respective coordinate arrays

In [ ]:
spot_locations = np.array(spot_locations.T.tolist() + bb_locations.T.tolist())

In [ ]:
zernike_calibrations = []
nz = 20
for i in range(len(spot_locations)):
    zernike_xy = np.flip(toolbox.convert_vector([1440+10,1024+65] - spot_locations[i],from_units="ij",to_units="zernike",hardware=fs).T[0])
    zernike_arr = np.concatenate([zernike_xy, np.zeros(nz -2, dtype=zernike_xy.dtype)])
    zernike_calibrations.append({i+1: float(v) for i, v in enumerate(zernike_arr)})
zernike_indices = [i+1 for i in range(nz)]

In [ ]:
phase = get_cholo_bb_g_phase(zernike_calibrations=zernike_calibrations,fs = fs,bb_phase = bottle_beam_phase(fs),n=60)

In [ ]:
plot_cam(fs=fs,phase = phase)